# Lab 5: Functional Python

Welcome to Lab 5 of Programming Methods! In this class we explore **functional programming** in Python: first-class functions, decorators, generators and lazy iteration, and higher-order (second-order) functions.

Create a new branch for this Lab (e.g., `git switch -c lab-05-functional`). Then, complete the following exercises in **this** Jupyter Notebook. Once you are done, **commit your work** using the proper Git commands.

Complete each exercise in the spaces provided below.

## Learning objectives

By the end of this class, you should be able to:

- Write custom decorators (`@timer`, `@retry`) to add timing and retry behavior to a function without changing its code.
- Use generators and `yield` to process a large dataset lazily, one item at a time, instead of loading it all into memory.
- Replace explicit loops with higher-order functions (`map`, `filter`, `functools.reduce`) to write clean, side-effect-free data transformations.

### Functions as objects, decorators, and lazy pipelines

So far your classes and objects have modeled *state* (API clients, pipeline stages, agents on a grid). Today's focus is different: treating **functions themselves as data** - passing them around, wrapping them, and chaining them lazily.

- **First-class functions**: functions can be assigned to variables, passed as arguments, returned from other functions, and stored in data structures - exactly like any other object.
- **Decorators**: functions that take a function and return a new function, used to add cross-cutting behavior (timing, retrying, logging) without changing the wrapped function's code.
- **Generators and `yield`**: functions that produce a sequence of values *lazily*, one at a time, instead of building the whole sequence in memory - essential when a dataset (or an LLM's streamed output) is too large to hold all at once.
- **Second-order (higher-order) functions**: functions like `map`, `filter`, and `functools.reduce` that take other functions as arguments, letting you express transformations declaratively instead of with explicit loops.

## Setup

Run this cell first.

In [1]:
import time
import functools
import itertools
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

# Make the local `pm_labs` package (../../src) importable from the notebook
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pm_labs import OllamaClient, WorldBankClient

# Change this to a model you have pulled locally, e.g. `ollama pull llama3.2:1b`
wb_client = WorldBankClient()
ollama_client = OllamaClient(model="llama3.2:1b")

---
## Exercise 1: Custom decorators - `@timer` and `@retry`

Calls to external services (a REST API, a local Ollama model) are exactly the kind of thing you want to **time** and **retry** without cluttering the calling code with `try/except` and `time.time()` everywhere. Decorators let you add that behavior once, and reuse it on any function.

Rather than reimplementing HTTP handling, we'll reuse the **`WorldBankClient`** and **`OllamaClient`** you built in Lab 3 (imported above from `pm_labs`) and simply wrap their methods with decorators.

### 1a. Build the `@timer` decorator

Write `timer(func)`, using `functools.wraps`, that prints `f"{func.__name__} took {elapsed:.3f}s"` and returns the wrapped function's result. Test it on a function that calls `time.sleep`.

In [ ]:
def timer(func):
    # TODO: fill with your code
    pass

@timer
def slow_add(a: int, b: int) -> int:
    """Pretend to do expensive work before adding two numbers."""
    time.sleep(0.2)
    return a + b

# TODO: call slow_add and inspect the output

### 1b. Build the `@retry` decorator

Write a plain decorator `retry(func)` (no arguments) that retries the wrapped function up to 3 times with a 0.5s delay if it raises, printing each failed attempt and re-raising after the last one. 

Apply `@retry` and `@timer` to `ask_ollama` and `fetch_population`, then run both a few times - notice how decorator **order** changes whether `@timer` measures one attempt or all the retries.

In [ ]:
def retry(func):
    # TODO: fill with your code
    pass

In [ ]:
def ask_ollama(prompt: str) -> str:
    # TODO: fill with your code
    pass


In [ ]:
@timer
@retry
def fetch_population(country_code: str) -> int:
    """Fetch a country's population via the reused WorldBankClient (Lab 3)."""
    return wb_client.get_population(country_code).value


print(fetch_population("pt"))

**On decorator order:** with `@timer` on top of `@retry`, `timer` wraps the *already-retrying* function, so it measures the total time across all retries - not just one attempt (e.g. all 3 tries of `ask_ollama` if the first two fail). Swapping the order (`@retry` on top of `@timer`) would time and print each individual attempt separately, since `timer` would then wrap the raw, non-retrying function. Decorators apply bottom-up, so the one closest to the function runs "innermost."

---
## Exercise 2: Generators for lazy, out-of-memory processing

Imagine sending a huge log file, one line per record, through an LLM for summarization - loading the whole file into a list first would be wasteful (or impossible, if it doesn't fit in memory). **Generators** let you produce one item at a time, on demand.

In [ ]:
# Generate a handful of unique reviews with a single real Ollama call - not
# 10,000 calls, since the point of this exercise is laziness, not LLM cost.
FALLBACK_REVIEWS = [
    "The delivery was fast and the packaging was great",
    "Terrible experience, the item arrived broken",
    "Average product, nothing special but works fine",
    "Excellent quality, will buy again",
    "Not worth the price, disappointed",
]

prompt = (
    "You are generating synthetic test data for a Python coding exercise. "
    "Write 8 short, one-sentence example product reviews for a generic "
    "online store, some enthusiastic and some unsatisfied. Output exactly "
    "one review per line, with no numbering, preamble, or extra commentary."
)
raw_lines = ollama_client.chat([{"role": "user", "content": prompt}]).splitlines()

# Small local models sometimes add a conversational preamble line (e.g. "Here
# are 8 reviews:") - drop lines that look like that instead of an actual
# review, since those would otherwise get "summarized" as nonsense later on.
unique_reviews = [
    line.strip() for line in raw_lines
    if line.strip() and not line.rstrip().endswith(":")
]

# Small local models occasionally refuse entirely or return too little usable
# text - fall back to a fixed list so the rest of the notebook never depends
# on model mood.
if len(unique_reviews) < 4:
    print("Ollama returned unusable output, falling back to a fixed review list.")
    unique_reviews = FALLBACK_REVIEWS

reviews = unique_reviews * 1250  # simulate a "massive" dataset: ~10,000 lines

# Save the generated reviews
REVIEWS_PATH = PROJECT_ROOT / "data" / "reviews.txt"
REVIEWS_PATH.parent.mkdir(parents=True, exist_ok=True)
REVIEWS_PATH.write_text("\n".join(reviews))


### 2a. A lazy line reader

Write a generator `read_lines(path)` that `yield`s one stripped line at a time, without reading the whole file into a list. Confirm it's lazy with `itertools.islice(read_lines("reviews.txt"), 3)`.

In [ ]:
def read_lines(path: Path):
    # TODO: fill with your code
    pass

# TODO: confirm it's lazy with itertools.islice

### 2b. A lazy processing pipeline

Write a generator `summarize_reviews(lines)` that yields `{"review": line, "summary": ollama_summarize(line)}` for each line, where `ollama_summarize` asks `ollama_client` for a 3-word summary. 

Pull the first 20 results into `records` with `itertools.islice` and display the first 5 as a table - confirm from the printed "sending to Ollama:" lines that only those 20 (not all 10,000) get processed.

In [ ]:
def ollama_summarize(text: str) -> str:
    # TODO: fill with your code
    pass


def summarize_reviews(lines):
    # TODO: fill with your code
    pass

pipeline = summarize_reviews(read_lines(REVIEWS_PATH))

# TODO: pull the first 20 results with itertools.islice
# TODO: display the first 5 results as a table

**Why this matters:** even though `reviews.txt` has 10,000 lines, only 20 ever reach `ollama_summarize` - you can confirm this from the "sending to Ollama:" output. Nothing is read from disk, stripped, or actually sent to your model beyond what `islice` consumes. If `read_lines` had returned a `list` instead of using `yield`, the entire file would be read (and, worse, if `summarize_reviews` also built a list, *every* line would be sent to Ollama) before you ever looked at a single result - which does not scale to files that don't fit in memory, or to a model/API you're paying (in time or money) per call for.

---
## Exercise 3: Higher-order functions - `map`, `filter`, `reduce`

Loops that filter and transform data can often be expressed more declaratively - and with less room for accidental side effects - using **second-order functions**: functions that take *other functions* as arguments.

### 3a. Replace a loop with `map`/`filter`

Write `titles_with_loop(records)` using explicit loops: keep reviews without "broken"/"disappointed", then uppercase their summaries. 

Rewrite it as `titles_with_functional(records)` using `filter` and `map`. 

Time both with `@timer` and confirm they return the same result - the timings should be close (that's expected, and matters for next class's parallel processing).

In [ ]:
def is_positive(r: dict) -> bool:
    return "broken" not in r["review"] and "disappointed" not in r["review"]

# TODO: define the method titles_with_loops with the proper decorator

# TODO: define the method titles_with_functional with the proper decorator


titles_loop = titles_with_loop(records)
titles_fn = titles_with_functional(records)

### 3b. Aggregate with `functools.reduce`

Use `functools.reduce` to count how many `records` are "positive" in a single expression, with no loop or intermediate list. 

How does it compare in readability to a loop or `sum(1 for r in records if is_positive(r))`?

In [ ]:
# TODO: Complete this exercise.
# Add your code below.


**On readability:** `functools.reduce` here is more *general* (it can accumulate into anything, not just a count) but arguably less *readable* than `sum(1 for r in records if is_positive(r))` for this specific case - Python's own style guide favors `sum`/comprehensions over `reduce` for simple accumulations, reserving `reduce` for accumulations that don't fit a built-in like `sum`, `max`, or `any`.

**So why prefer `map`/`filter` at all, if the timing is basically the same?** On a single core it is, and that's expected - `map`/`filter` are not a performance optimization over a loop in CPython. Two reasons to still prefer them:

- **Intent.** `filter(is_positive, records)` reads as "keep only the positive ones" - a loop with an `if` and an `append` requires the reader to simulate the loop to see the same thing. It also removes a common source of bugs: forgetting to initialize the accumulator list, or mutating the wrong variable inside the loop.
- **It sets up parallel processing (next class).** `titles_with_functional`'s core step is just `map(transform_fn, iterable)` - and that's exactly the call signature of `concurrent.futures.ProcessPoolExecutor.map` and `multiprocessing.Pool.map`. A function already written this way can be parallelized across cores by swapping *which* `map` you call, with no change to `transform_fn` or `is_positive` themselves. `titles_with_loop`'s hand-written `for` loop has no equivalent one-line swap - it would need to be restructured first. That's the real payoff of today's timing being a tie: the functional version is *parallelization-ready* even though it isn't faster yet.

## Wrap-up questions

### 1. What does `functools.wraps` do?

A. It prevents the decorated function from being called more than once.  
B. It copies metadata such as `__name__` and `__doc__` from the original function to the wrapper.  
C. It automatically calls the original function inside the wrapper.  
D. It makes the decorator execute faster.  

---

### 2. What would happen if `read_lines` returned a list instead of yielding lines?

A. The file would be loaded entirely into memory before any lines could be processed.  
B. The function would only be able to return the first line of the file.  
C. Python would automatically convert the list into a generator.  
D. The file would be read more slowly, but memory usage would remain the same.  

---

### 3. What is the main difference between using `filter` and a list comprehension for filtering values?

A. `filter` can filter any iterable, while list comprehensions only work with lists.  
B. A list comprehension is always faster than `filter`.  
C. Both can produce the same filtered values, but comprehensions are often clearer for simple inline conditions, while `filter` can be useful with a named reusable predicate.  
D. `filter` modifies the original iterable, while a list comprehension creates a new one.  

---

### 4. Is `ollama_summarize` a pure function?

A. Yes, because it always receives its input as function arguments.  
B. Yes, because calling an external model does not affect function purity.  
C. No, because it interacts with an external model and performs a side effect by printing.  
D. No, but only because it uses `print`.  


> #### 📚 Further reading
> - Ramalho, *Fluent Python* (2nd ed.), **Chapter 7 "Functions as First-Class Objects"** — functions as objects that can be passed, returned, and stored (the foundation `timer` and `retry` rely on), plus the section on `map`, `filter`, and `reduce` vs. list comprehensions/generator expressions, including why `reduce` fell out of favor as a builtin in Python 3.
> - Ramalho, **Chapter 9 "Decorators and Closures"** — covers exactly the pattern used in Exercise 1: `functools.wraps` (relevant to wrap-up question 1) and closures (how `wrapper` remembers `func` between calls).
> - Ramalho, **Chapter 17 "Iterators, Generators, and Classic Coroutines"** — the full mechanics of `yield`, generator functions vs. generator expressions, and why chaining generators keeps everything lazy, as in Exercise 2.
